# Face Mask and Emotion Detection - EDA and Preprocessing

in this notebook i will:
- load the two datasets (face mask and fer2013)
- clean the data (remove duplicates, fix corrupted images)
- explore the data with some charts
- handle class imbalance

the two datasets i used:
- face mask dataset from kaggle (andrewmvd)
- fer2013 emotion dataset from kaggle

## install libraries

In [ ]:
# first install what we need
!pip install opendatasets albumentations opencv-python-headless -q

In [ ]:
# import everything
import os
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import xml.etree.ElementTree as ET
from pathlib import Path
from PIL import Image, UnidentifiedImageError
from collections import Counter
from sklearn.utils.class_weight import compute_class_weight
import albumentations as A
import warnings
warnings.filterwarnings('ignore')

print('all imports done')

## Step 1 - Download and Load the Datasets

In [ ]:
# download datasets from kaggle
# you need to enter your kaggle username and key when asked
# get the key from kaggle.com -> settings -> API -> Create New Token

import opendatasets as od

od.download('https://www.kaggle.com/datasets/andrewmvd/face-mask-detection')
od.download('https://www.kaggle.com/datasets/msambare/fer2013')

print('datasets downloaded')

In [ ]:
# set folder paths
MASK_IMG_DIR  = Path('face-mask-detection/images')
MASK_ANNO_DIR = Path('face-mask-detection/annotations')
FER_DIR       = Path('fer2013')

EMOTION_LABELS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

print('mask images folder:', MASK_IMG_DIR)
print('fer2013 folder:', FER_DIR)

In [ ]:
# the mask dataset uses xml annotation files
# each xml file has bounding boxes and labels (with_mask, without_mask, mask_weared_incorrect)
# i will read them and make a dataframe

def read_mask_xml(anno_dir):
    rows = []
    # map all labels to just 2 classes: with_mask=1, without_mask=0
    label_map = {
        'with_mask': 1,
        'without_mask': 0,
        'mask_weared_incorrect': 0
    }
    for xml_file in sorted(anno_dir.glob('*.xml')):
        tree = ET.parse(xml_file)
        root = tree.getroot()
        filename = root.find('filename').text
        for obj in root.findall('object'):
            label_str = obj.find('name').text.strip().lower()
            label = label_map.get(label_str, 0)
            rows.append({'filename': filename, 'label': label, 'label_name': label_str})
    return pd.DataFrame(rows)

mask_df = read_mask_xml(MASK_ANNO_DIR)
print('mask dataset shape:', mask_df.shape)
print(mask_df['label_name'].value_counts())
mask_df.head()

In [ ]:
# for fer2013, the folder structure is already train/emotion_name/
# so i just need to get all file paths and labels

def load_fer_paths(fer_dir, split='train'):
    rows = []
    for label_idx, emotion in enumerate(EMOTION_LABELS):
        folder = fer_dir / split / emotion
        if not folder.exists():
            continue
        for img_path in folder.glob('*.jpg'):
            rows.append({
                'filepath': str(img_path),
                'label': label_idx,
                'emotion': emotion
            })
    return pd.DataFrame(rows)

fer_train = load_fer_paths(FER_DIR, 'train')
fer_test  = load_fer_paths(FER_DIR, 'test')
fer_df    = pd.concat([fer_train, fer_test], ignore_index=True)

print('fer2013 total images:', len(fer_df))
print(fer_df['emotion'].value_counts())
fer_df.head()

## Step 2 - Remove Duplicate Images

i will use MD5 hashing to find images that are exactly the same.
if two images have the same hash, they are identical, so i will keep only one.

In [ ]:
def get_md5(filepath):
    # read the image bytes and compute a hash
    try:
        with open(filepath, 'rb') as f:
            return hashlib.md5(f.read()).hexdigest()
    except:
        return None

# check duplicates in fer2013
print('checking for duplicates in fer2013...')
fer_df['hash'] = fer_df['filepath'].apply(get_md5)

before = len(fer_df)
fer_df.drop_duplicates(subset='hash', inplace=True)
after = len(fer_df)

print(f'before: {before} images')
print(f'after removing duplicates: {after} images')
print(f'removed: {before - after} duplicate images')

In [ ]:
# check duplicates in mask dataset
mask_img_list = list(MASK_IMG_DIR.glob('*.png')) + list(MASK_IMG_DIR.glob('*.jpg'))

seen_hashes = {}
duplicate_count = 0

for img_path in mask_img_list:
    h = get_md5(img_path)
    if h in seen_hashes:
        duplicate_count += 1
    else:
        seen_hashes[h] = img_path

print(f'mask dataset: {len(mask_img_list)} total images')
print(f'duplicates found: {duplicate_count}')

## Step 3 - Handle Corrupted or Missing Files

some images in the dataset might be broken or missing.
i will check each image and if it cant be opened i will remove it.
if less than 5% are broken its safe to just drop them.

In [ ]:
def is_valid_image(filepath):
    # try opening the image, if it fails its corrupted
    try:
        img = Image.open(filepath)
        img.verify()
        return True
    except:
        return False

print('checking image integrity... (this takes a moment)')
fer_df['is_valid'] = fer_df['filepath'].apply(is_valid_image)

corrupted = fer_df[~fer_df['is_valid']]
pct = len(corrupted) / len(fer_df) * 100

print(f'corrupted images: {len(corrupted)} ({pct:.2f}%)')

if pct < 5:
    print('less than 5% corrupted -> safe to drop them')
    fer_df = fer_df[fer_df['is_valid']].reset_index(drop=True)
    print(f'clean dataset size: {len(fer_df)}')
else:
    print('WARNING: more than 5% corrupted, need to investigate!')

## Step 4 - Detect Outlier Images (blank or wrong size)

outliers can be:
- images with very low pixel variation (basically blank)
- images with unusual sizes

i will look at a sample and visualize what outliers look like vs normal images.

In [ ]:
def get_img_info(filepath):
    # get width, height, and pixel std (low std = blank image)
    try:
        img = cv2.imread(str(filepath))
        if img is None:
            return None
        h, w = img.shape[:2]
        std_px = img.std()  # how much the pixels vary
        return w, h, std_px
    except:
        return None

# check a sample of 500 images
sample = fer_df['filepath'].sample(500, random_state=42).tolist()
info_list = [get_img_info(p) for p in sample]
info_list = [x for x in info_list if x is not None]

widths = [x[0] for x in info_list]
heights = [x[1] for x in info_list]
stds = [x[2] for x in info_list]

# images with std < 5 are basically blank
blank_count = sum(1 for s in stds if s < 5)
print(f'possibly blank images in sample: {blank_count}')
print(f'most common sizes:', Counter(zip(widths, heights)).most_common(3))

In [ ]:
# show what normal vs blank images look like
normal_paths = [p for p, s in zip(sample, stds) if s >= 10][:4]
blank_paths  = [p for p, s in zip(sample, stds) if s < 10][:4]

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
fig.suptitle('Step 4 - Normal Images vs Outlier Images', fontsize=13, fontweight='bold')

for i, p in enumerate(normal_paths):
    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    axes[0][i].imshow(img)
    axes[0][i].set_title('normal')
    axes[0][i].axis('off')

for i, p in enumerate(blank_paths + [None] * 4):  # pad if not enough blanks
    if p:
        img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        axes[1][i].imshow(img)
    else:
        axes[1][i].imshow(np.zeros((48, 48, 3), dtype=np.uint8))
    axes[1][i].set_title('outlier')
    axes[1][i].axis('off')

plt.tight_layout()
os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/step4_outliers.png')
plt.show()
print('saved to outputs/step4_outliers.png')

## Step 5 - EDA and Visualizations

now i will make some charts to understand the data better.
i will make 7 plots total.

In [ ]:
# plot 1 - class distribution for both datasets

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Plot 1 - Class Distribution', fontsize=13, fontweight='bold')

# mask dataset
mask_counts = mask_df['label_name'].value_counts()
mask_counts.plot(kind='bar', ax=axes[0], color=['green', 'red', 'orange'], edgecolor='black')
axes[0].set_title('Face Mask Dataset')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=20)
for p in axes[0].patches:
    axes[0].text(p.get_x() + p.get_width()/2, p.get_height() + 10,
                 str(int(p.get_height())), ha='center', fontsize=9)

# emotion dataset
emo_counts = fer_df['emotion'].value_counts()
emo_counts.plot(kind='bar', ax=axes[1], color=sns.color_palette('Set2', 7), edgecolor='black')
axes[1].set_title('FER2013 Emotion Dataset')
axes[1].set_xlabel('Emotion')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=30)
for p in axes[1].patches:
    axes[1].text(p.get_x() + p.get_width()/2, p.get_height() + 10,
                 str(int(p.get_height())), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('outputs/plot1_class_distribution.png')
plt.show()
print('plot 1 done')

In [ ]:
# plot 2 - sample images for each emotion class
fig, axes = plt.subplots(7, 5, figsize=(12, 18))
fig.suptitle('Plot 2 - Sample Images per Emotion', fontsize=13, fontweight='bold')

for row, emotion in enumerate(EMOTION_LABELS):
    subset = fer_df[fer_df['emotion'] == emotion]['filepath'].sample(5, random_state=42)
    for col, path in enumerate(subset):
        img = cv2.imread(path)
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        else:
            img = np.zeros((48, 48, 3), dtype=np.uint8)
        axes[row][col].imshow(img)
        axes[row][col].axis('off')
        if col == 0:
            axes[row][col].set_ylabel(emotion, fontsize=9, fontweight='bold',
                                       rotation=0, labelpad=40, va='center')

plt.tight_layout()
plt.savefig('outputs/plot2_sample_grid.png')
plt.show()
print('plot 2 done')

In [ ]:
# plot 3 - image size distribution
# how big are the images? are they all the same size?

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Plot 3 - Image Size Distribution', fontsize=13, fontweight='bold')

axes[0].hist(widths, bins=20, color='steelblue', edgecolor='black')
axes[0].set_title('Width')
axes[0].set_xlabel('pixels')
axes[0].set_ylabel('count')

axes[1].hist(heights, bins=20, color='salmon', edgecolor='black')
axes[1].set_title('Height')
axes[1].set_xlabel('pixels')

aspect_ratios = [w/h for w, h in zip(widths, heights)]
axes[2].hist(aspect_ratios, bins=20, color='mediumpurple', edgecolor='black')
axes[2].set_title('Aspect Ratio (W/H)')
axes[2].set_xlabel('ratio')
axes[2].axvline(1.0, color='red', linestyle='--', label='1:1 (square)')
axes[2].legend()

plt.tight_layout()
plt.savefig('outputs/plot3_size_distribution.png')
plt.show()
print('plot 3 done')

In [ ]:
# plot 4 - pixel intensity for each color channel (R, G, B)
# this shows if the dataset is mostly dark or bright

R_vals, G_vals, B_vals = [], [], []

for path in fer_df['filepath'].sample(200, random_state=42):
    img = cv2.imread(path)
    if img is None:
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (48, 48))
    R_vals.extend(img[:, :, 0].flatten().tolist())
    G_vals.extend(img[:, :, 1].flatten().tolist())
    B_vals.extend(img[:, :, 2].flatten().tolist())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Plot 4 - Pixel Intensity Histograms', fontsize=13, fontweight='bold')

for ax, vals, color, label in zip(axes, [R_vals, G_vals, B_vals],
                                   ['red', 'green', 'blue'], ['Red', 'Green', 'Blue']):
    ax.hist(vals, bins=50, color=color, alpha=0.7, edgecolor='none')
    ax.set_title(f'{label} channel')
    ax.set_xlabel('pixel value (0-255)')
    ax.set_ylabel('frequency')
    ax.axvline(np.mean(vals), color='black', linestyle='--',
               label=f'mean = {np.mean(vals):.0f}')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('outputs/plot4_pixel_intensity.png')
plt.show()
print('plot 4 done')

In [ ]:
# plot 5 - pie charts for class percentages

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Plot 5 - Class Percentages', fontsize=13, fontweight='bold')

mask_c = mask_df['label_name'].value_counts()
axes[0].pie(mask_c, labels=mask_c.index, autopct='%1.1f%%',
            colors=['green', 'red', 'orange'],
            wedgeprops={'edgecolor': 'white'})
axes[0].set_title('Mask Dataset')

emo_c = fer_df['emotion'].value_counts()
axes[1].pie(emo_c, labels=emo_c.index, autopct='%1.1f%%',
            colors=sns.color_palette('Set2', len(emo_c)),
            wedgeprops={'edgecolor': 'white'})
axes[1].set_title('Emotion Dataset')

plt.tight_layout()
plt.savefig('outputs/plot5_class_percentages.png')
plt.show()
print('plot 5 done')

In [ ]:
# plot 6 - average face per emotion
# i average all the pixel values for each emotion to see what the "typical" face looks like

fig, axes = plt.subplots(1, 7, figsize=(16, 3))
fig.suptitle('Plot 6 - Average Face Per Emotion Class', fontsize=13, fontweight='bold')

for ax, emotion in zip(axes, EMOTION_LABELS):
    paths = fer_df[fer_df['emotion'] == emotion]['filepath'].head(100).tolist()
    images = []
    for p in paths:
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            images.append(cv2.resize(img, (48, 48)).astype('float32'))
    if images:
        mean_face = np.mean(images, axis=0).astype('uint8')
        ax.imshow(mean_face, cmap='gray')
    ax.set_title(emotion, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('outputs/plot6_average_faces.png')
plt.show()
print('plot 6 done')

In [ ]:
# plot 7 - correlation heatmap between pixel regions and emotion label
# i flatten each image to small 8x8 pixels and check how correlated each region is with the label

feature_rows = []
for p, emo in zip(fer_df['filepath'].head(400), fer_df['emotion'].head(400)):
    img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue
    resized = cv2.resize(img, (8, 8)).flatten()
    feature_rows.append(list(resized) + [EMOTION_LABELS.index(emo)])

feat_df = pd.DataFrame(feature_rows,
                        columns=[f'px{i}' for i in range(64)] + ['label'])

# pick every 8th pixel plus the label to keep the heatmap readable
cols = [f'px{i}' for i in range(0, 64, 8)] + ['label']
corr = feat_df[cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax,
            linewidths=0.5, square=True, annot_kws={'size': 8})
ax.set_title('Plot 7 - Pixel Region Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/plot7_correlation_heatmap.png')
plt.show()
print('plot 7 done')

## Step 6 - Handle Class Imbalance

some emotions appear much more than others (happy is common, disgust is rare).
i will handle this in two ways:
1. compute class weights so the model pays more attention to rare classes
2. use data augmentation with albumentations to create more variety

In [ ]:
# show class percentages clearly
print('=== FER2013 class distribution ===')
emo_val = fer_df['emotion'].value_counts()
for emo, count in emo_val.items():
    pct = count / len(fer_df) * 100
    print(f'  {emo:10s}: {count:5d}  ({pct:.1f}%)')

print('\n=== mask dataset class distribution ===')
mask_val = mask_df['label_name'].value_counts()
for cls, count in mask_val.items():
    pct = count / len(mask_df) * 100
    print(f'  {cls:25s}: {count:4d}  ({pct:.1f}%)')

In [ ]:
# compute class weights for the emotion classes
# these weights will be passed to model.fit() so rarer classes get more importance

all_labels = fer_df['label'].values
unique_labels = np.unique(all_labels)

weights = compute_class_weight('balanced', classes=unique_labels, y=all_labels)
class_weights = dict(zip(unique_labels, weights))

print('class weights for emotion model:')
for k, v in class_weights.items():
    print(f'  class {k} ({EMOTION_LABELS[k]}): weight = {v:.4f}')

# save for use in training notebook
print('\nwe will pass this dict to model.fit(class_weight=class_weights)')

In [ ]:
# define albumentations augmentation pipeline
# this will generate modified versions of training images

augment = A.Compose([
    A.HorizontalFlip(p=0.5),           # flip left-right
    A.Rotate(limit=15, p=0.4),          # small rotation
    A.RandomBrightnessContrast(p=0.4),  # change brightness/contrast
    A.GaussNoise(var_limit=(5, 25), p=0.2),  # add noise
    A.Resize(224, 224),                 # resize for model input
])

# show 5 augmented versions of one image
sample_img = cv2.cvtColor(cv2.imread(fer_df['filepath'].iloc[0]), cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 6, figsize=(16, 3))
axes[0].imshow(sample_img)
axes[0].set_title('original')
axes[0].axis('off')

for i in range(1, 6):
    aug_img = augment(image=sample_img)['image']
    axes[i].imshow(aug_img)
    axes[i].set_title(f'aug {i}')
    axes[i].axis('off')

plt.suptitle('Augmented Image Examples', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/plot6b_augmentation_examples.png')
plt.show()
print('augmentation examples saved')

In [ ]:
# show before/after oversampling - just to illustrate the concept
before_counts = fer_df['emotion'].value_counts().copy()

minority_class = before_counts.idxmin()
minority_rows  = fer_df[fer_df['emotion'] == minority_class]

# oversample the minority class by duplicating it
oversampled_df = pd.concat([fer_df, minority_rows.sample(len(minority_rows),
                                                          replace=True, random_state=42)])
after_counts = oversampled_df['emotion'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Class Distribution Before vs After Oversampling', fontsize=13, fontweight='bold')

before_counts.plot(kind='bar', ax=axes[0], color=sns.color_palette('Set2', 7), edgecolor='black')
axes[0].set_title('before')
axes[0].tick_params(axis='x', rotation=30)

after_counts.plot(kind='bar', ax=axes[1], color=sns.color_palette('Set2', 7), edgecolor='black')
axes[1].set_title(f'after (oversampled: {minority_class})')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('outputs/plot6c_before_after_oversampling.png')
plt.show()
print('done')

## Summary

what i did in this notebook:

| step | what i did |
|------|------------|
| 1 | loaded face mask dataset (from XML) and FER2013 dataset |
| 2 | found and removed duplicate images using MD5 hashing |
| 3 | checked for corrupted files and dropped them (less than 5%) |
| 4 | found outlier images (blank/wrong size) and visualized them |
| 5 | made 7 plots to explore the data |
| 6 | computed class weights and set up augmentation pipeline |

next step -> open notebook 02 to train the models